### Import Library

In [ ]:
import pandas as pd

### Import CSV Files

In [ ]:
hybrid_file = f'Hybrid/metrics_hybrid.csv'
metrics_hybrid = pd.read_csv(hybrid_file, sep = ';')

#### Colunas

In [ ]:
columns_hybrid = ["shot", "found", "found_weight", "original_weight", "degenerate", "is_equal", "size_active_positions", "total_tested_combinations", 
                   "hybrid", "iteration", "time_sum_ms", "total_time_search_ms", "transfer_time_ms", "total_to_find_ms" ]


In [ ]:
hibrido_aux_prob = metrics_hybrid[columns_hybrid].copy()

df = hibrido_aux_prob.copy()

df

Número de padrões em comum: 100000


,pattern_id,found,found_weight,degenerate,iteration,tam,total_tested_combinations,hibrido,time_sum_ms,total_time_search_ms,transfer_time_ms,total_to_find_ms
0,0,0,0,0,3,17,65331,0,0.043200,0.079744,0.047518,0.0
1,1,0,0,0,3,12,7527,0,0.006080,0.121856,0.034671,0.0
2,2,0,0,0,3,17,65331,0,0.007296,0.220000,0.034983,0.0
3,3,0,0,0,3,27,1192779,0,0.007712,0.170880,0.035793,0.0
4,4,0,0,0,3,10,2541,-1,0.008416,0.487639,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99995,0,0,0,3,19,131385,0,0.007808,0.132768,0.033116,0.0
99996,99996,0,0,0,3,88,1167107569,0,0.007392,149.083710,0.037294,0.0
99997,99997,0,0,0,3,89,1250408109,0,0.008384,54.292801,0.036981,0.0
99998,99998,0,0,0,3,12,7527,0,0.008896,0.070432,0.034819,0.0


Fração do espaço explorado pela CPU

In [ ]:
def assign_tam_group(size_active_positions):
    if size_active_positions == 1:
        return "1"
    if size_active_positions == 2:
        return "2"
    if size_active_positions <= 5:
        return "3-5"
    elif size_active_positions <= 10:
        return "6-10"
    elif size_active_positions <= 15:
        return "11-15"
    elif size_active_positions <= 20:
        return "16-20"
    elif size_active_positions <= 25:
        return "21-25"
    elif size_active_positions <= 30:
        return "26-30"
    elif size_active_positions <= 35:
        return "31-35"
    elif size_active_positions <= 40:
        return "36-40"
    elif size_active_positions <= 45:
        return "41-45"
    elif size_active_positions <= 50:
        return "46-50"
    elif size_active_positions <= 55:
        return "51-55"
    elif size_active_positions <= 60:
        return "56-60"
    elif size_active_positions <= 65:
        return "61-65"
    elif size_active_positions <= 70:
        return "66-70"
    elif size_active_positions <= 75:
        return "71-75"
    elif size_active_positions <= 80:
        return "76-80"
    elif size_active_positions <= 85:
        return "81-85"
    elif size_active_positions <= 90:
        return "86-90"
    elif size_active_positions <= 95:
        return "91-95"
    else:
        return "96+"

def tam_group_to_upper_bound(group_str):
    group_str = str(group_str).strip()
    if "+" in group_str:
        return int(group_str.replace("+", ""))
    elif "-" in group_str:
        return int(group_str.split("-")[1])
    else:
        return int(group_str)

def get_phase_probabilities_fixed(row, max_weight_decoder=7):

    size_active_positions = int(row["size_active_positions"])
    max_feasible = min(max_weight_decoder, size_active_positions)

    def p(w):
        return row.get(f"p{w}", 0.0) if w <= max_feasible else 0.0

    prob_0 = sum(p(w) for w in [1, 2, 3, 4, 5])
    prob_1 = p(6)
    prob_2 = p(7)


    return pd.Series({ "prob_phase0": prob_0, "prob_phase1": prob_1, "prob_phase2": prob_2})

def get_phase_order_by_probability(row):
    phase_probs = {
        0: row["prob_phase0"],   # {1,2,3,4,5}
        1: row["prob_phase1"],   # {6}
        2: row["prob_phase2"],   # {7}
    }

    ordered = sorted(phase_probs.items(), key=lambda x: x[1], reverse=True)
    ordered_ids = [phase_id for phase_id, _ in ordered]

    return pd.Series({ "order0": ordered_ids[0], "order1": ordered_ids[1],"order2": ordered_ids[2] })

In [ ]:
df_valid = df[df["found"] == 1].copy()

counts = ( df_valid.groupby(["size_active_positions", "found_weight"]).size().reset_index(name="count"))

total_per_tam = ( df_valid.groupby("size_active_positions") .size() .reset_index(name="total") )

stats = counts.merge(total_per_tam, on="size_active_positions")
stats["prob"] = stats["count"] / stats["total"]

pivot = stats.pivot(index="size_active_positions", columns="found_weight", values="prob").fillna(0)
pivot.columns = [f"p{int(c)}" for c in pivot.columns]
pivot = pivot.reset_index()

final = pivot.merge(total_per_tam, on="size_active_positions")
final["size_active_positions_group"] = final["size_active_positions"].apply(assign_tam_group)

prob_cols = [c for c in final.columns if c.startswith("p")]

rows = []
for group_name, group in final.groupby("size_active_positions_group", sort=False):
    row = { "size_active_positions_group": group_name, "total": group["total"].sum() }

    total_sum = group["total"].sum()

    for col in prob_cols:
        row[col] = (group[col] * group["total"]).sum() / total_sum

    rows.append(row)

grouped_df = pd.DataFrame(rows)
grouped_df["size_active_positions"] = grouped_df["size_active_positions_group"].apply(tam_group_to_upper_bound)

phase_probs = grouped_df.apply(get_phase_probabilities_fixed, axis=1)
grouped_df = pd.concat([grouped_df, phase_probs], axis=1)

phase_order = grouped_df.apply(get_phase_order_by_probability, axis=1)
grouped_df = pd.concat([grouped_df, phase_order], axis=1)

grouped_df[[ "size_active_positions_group", "size_active_positions", "total", "prob_phase0", "prob_phase1", "prob_phase2", "order0", "order1", "order2" ]]

,tam_group,tam,total,prob_phase0,prob_phase1,prob_phase2,order0,order1,order2
0,1,1,23,1.000000,0.000000,0.0,0,1,2
1,2,2,102,1.000000,0.000000,0.0,0,1,2
2,3-5,5,1618,1.000000,0.000000,0.0,0,1,2
3,6-10,10,5847,0.537370,0.462630,0.0,0,1,2
4,11-15,15,1813,0.291782,0.708218,0.0,1,0,2
5,16-20,20,423,0.345154,0.654846,0.0,1,0,2
6,21-25,25,228,0.561404,0.438596,0.0,0,1,2
7,26-30,30,178,0.617978,0.382022,0.0,0,1,2
8,31-35,35,219,0.625571,0.374429,0.0,0,1,2
9,36-40,40,255,0.588235,0.411765,0.0,0,1,2


In [ ]:
with open(f"probabilities.txt", "w", encoding="utf-8") as f:
    for _, row in grouped_df.iterrows():
        size_upper = int(row["size_active_positions"])

        values = [ size_upper, int(row["order0"]), int(row["order1"]), int(row["order2"]), row["prob_phase0"], row["prob_phase1"], row["prob_phase2"], ]

        line = " ".join( f"{x:.6f}" if isinstance(x, float) else str(x) for x in values )
        f.write(line + "\n")